In [16]:
import numpy as np
import pandas as pd
from lenskit import batch, topn, util
import pickle
from collections import defaultdict

In [2]:
PROCESSED_FILE_DIR = "/home/shoaib/recommender-system/royal/journal-revision/New/FH/processed-files"
DATA_FILE_DIR = "/home/shoaib/recommender-system/royal/journal-revision/New/FH/data"

In [3]:
# Specify the path to your pickle file
pickle_file_path = f"{PROCESSED_FILE_DIR}/TrustWorthy_FH_MAGrec.pickle"

# Open the pickle file in read binary mode
with open(pickle_file_path, 'rb') as file:
    # Load the data from the pickle file
    loaded_data = pickle.load(file)

In [4]:
history_u_lists, history_ur_lists, history_v_lists, history_vr_lists, train_u, train_v, train_r, test_u, test_v, test_r, social_adj_lists, ratings_list=loaded_data

In [5]:
social_adj_lists

defaultdict(set,
            {0: {1290},
             2: {610, 1632},
             18: {891},
             22: {1194},
             25: {1545, 1565, 3315},
             26: {300},
             37: {15, 679},
             47: {24, 2476, 3978},
             49: {501},
             51: {421},
             57: {671, 964, 1337},
             60: {396},
             64: {610},
             65: {373, 396, 518},
             67: {582},
             70: {86},
             71: {1632, 2254},
             85: {683, 785},
             87: {161},
             92: {1632},
             96: {2024, 2045, 2336},
             97: {21, 450},
             98: {891, 3992},
             103: {2257},
             104: {679, 4160},
             108: {87, 252, 962, 1041, 1498},
             111: {636},
             113: {5246},
             120: {891},
             123: {160},
             127: {344, 638, 1717, 2303},
             128: {1333},
             129: {1195, 1328, 2077},
             131: {57},
       

In [10]:
# Specify the path to your pickle file
untrusted_users_pickle_file_path = f"{PROCESSED_FILE_DIR}/untrusted_users_ss40.pickle"

# Open the pickle file in read binary mode
with open(untrusted_users_pickle_file_path, 'rb') as file:
    # Load the data from the pickle file
    loaded_data = pickle.load(file)

untrusted_users_ss60 = loaded_data

In [11]:
untrusted_users_ss60

[1,
 6,
 18,
 46,
 80,
 97,
 101,
 108,
 111,
 136,
 142,
 144,
 164,
 207,
 214,
 264,
 267,
 358,
 388,
 410,
 417,
 432,
 454,
 464,
 469,
 473,
 483,
 487,
 502,
 503,
 525,
 558,
 561,
 573,
 639,
 655,
 658,
 667,
 703,
 733,
 743,
 750,
 756,
 766,
 775,
 785,
 804,
 852,
 862,
 869,
 880,
 883,
 899,
 917,
 922,
 966,
 973,
 982,
 983,
 1025,
 1038,
 1049,
 1082,
 1094,
 1104,
 1113,
 1136,
 1159,
 1161,
 1173,
 1186,
 1197,
 1198,
 1231,
 1253,
 1284,
 1301,
 1305,
 1348,
 1350,
 1371,
 1395,
 1406,
 1476,
 1477,
 1484,
 1500,
 1509,
 1534,
 1544,
 1576,
 1581,
 1600,
 1603,
 1609,
 1646,
 1654,
 1664,
 1666,
 1670,
 1672,
 1693,
 1712,
 1718,
 1723,
 1737,
 1739,
 1777,
 1796,
 1801,
 1814,
 1819,
 1834,
 1848,
 1864,
 1873,
 1874,
 1876,
 1880,
 1891,
 1906,
 1910,
 1911,
 1951,
 1957,
 1968,
 2061,
 2074,
 2078,
 2081,
 2153,
 2231,
 2251,
 2270,
 2272,
 2351,
 2369,
 2377,
 2392,
 2396,
 2465,
 2473,
 2487,
 2495,
 2496,
 2513,
 2544,
 2556,
 2569,
 2584,
 2595,
 2598,
 26

In [12]:
# Removes the first occurrence of 5404 in-place
untrusted_users_ss60.remove(5404)

In [17]:
# Create a new defaultdict to store modified values
new_dict = defaultdict(set)

# Remove values in untrusted_users_ss60 from the original defaultdict
for key, values in social_adj_lists.items():
    values -= set(untrusted_users_ss60)
    if values:  # Add the key to the new defaultdict only if the set is not empty
        new_dict[key] = values

In [18]:
import pickle
# Organize the variables into a tuple in the desired order
data_to_save = (history_u_lists, history_ur_lists, history_v_lists, history_vr_lists,
                train_u, train_v, train_r, test_u, test_v, test_r, new_dict, ratings_list)

# Specify the path to save the pickle file
output_pickle_file_path = f"{PROCESSED_FILE_DIR}/FH_Trustworthy_Social_0.6_threshold.pickle"


# Open the pickle file in write binary mode ('wb')
with open(output_pickle_file_path, 'wb') as file:
    # Dump the data to the pickle file
    pickle.dump(data_to_save, file)

print("Data saved to pickle file:", output_pickle_file_path)

Data saved to pickle file: /home/shoaib/recommender-system/royal/journal-revision/New/FH/processed-files/FH_Trustworthy_Social_0.6_threshold.pickle


## Training

In [19]:
import torch
import torch.nn as nn
import pickle
import numpy as np
import argparse
import os
from UV_Encoders import UV_Encoder
from UV_Aggregators import UV_Aggregator
from Social_Encoders import Social_Encoder
from Social_Aggregators import Social_Aggregator


class GraphRec(nn.Module):
    def __init__(self, enc_u, enc_v_history, r2e):
        super(GraphRec, self).__init__()
        self.enc_u = enc_u
        self.enc_v_history = enc_v_history
        self.embed_dim = enc_u.embed_dim
        self.w_ur1 = nn.Linear(self.embed_dim, self.embed_dim)
        self.w_ur2 = nn.Linear(self.embed_dim, self.embed_dim)
        self.w_vr1 = nn.Linear(self.embed_dim, self.embed_dim)
        self.w_vr2 = nn.Linear(self.embed_dim, self.embed_dim)
        self.w_uv1 = nn.Linear(self.embed_dim * 2, self.embed_dim)
        self.w_uv2 = nn.Linear(self.embed_dim, 16)
        self.w_uv3 = nn.Linear(16, 1)
        self.r2e = r2e
        self.bn1 = nn.BatchNorm1d(self.embed_dim, momentum=0.5)
        self.bn2 = nn.BatchNorm1d(self.embed_dim, momentum=0.5)
        self.bn3 = nn.BatchNorm1d(self.embed_dim, momentum=0.5)
        self.bn4 = nn.BatchNorm1d(16, momentum=0.5)

    def forward(self, nodes_u, nodes_v):
        embeds_u = self.enc_u(nodes_u)
        embeds_v = self.enc_v_history(nodes_v)
        scores_u = torch.matmul(embeds_u, embeds_v.t())
        return scores_u.squeeze()

    def bpr_loss(self, nodes_u, nodes_v, device, num_items):
        scores_u = self.forward(nodes_u, nodes_v)
        nodes_v_negative = torch.randint(0, num_items, size=nodes_u.size(), dtype=torch.long).to(device)
        scores_u_negative = self.forward(nodes_u, nodes_v_negative)
        bpr_loss = -torch.log(torch.sigmoid(scores_u - scores_u_negative)).mean()
        return bpr_loss

def train(model, device, train_loader, optimizer, epoch, num_items):
    model.train()
    running_loss = 0.0
    for i, data in enumerate(train_loader, 0):
        batch_nodes_u, batch_nodes_v, _ = data
        optimizer.zero_grad()
        loss = model.bpr_loss(batch_nodes_u.to(device), batch_nodes_v.to(device), device, num_items)
        loss.backward(retain_graph=True)
        optimizer.step()
        running_loss += loss.item()
        if i % 100 == 0:
            print('[%d, %5d] loss: %.3f' % (epoch, i, running_loss / 100))
            running_loss = 0.0

def get_top_100_recommendations(model, device, test_loader, num_items, k=100):
    model.eval()
    user_recommendations_dict = {}  # Dictionary to store user recommendations

    with torch.no_grad():
        for test_u, test_v, _ in test_loader:
            test_u, test_v = test_u.to(device), test_v.to(device)
            scores = model.forward(test_u, test_v)
            scores = scores.squeeze().tolist()
            top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]

            user_ids = test_u.tolist()  # Convert tensor to list

            for user_id in user_ids:
                user_recommendations = [(rank, test_v[item_index].item()) for rank, item_index in enumerate(top_indices, start=1)]
                user_recommendations_dict[user_id] = user_recommendations

    return user_recommendations_dict




def main():
    parser = argparse.ArgumentParser(description='Social Recommendation: GraphRec model')
    parser.add_argument('--batch_size', type=int, default=128, metavar='N', help='input batch size for training')
    parser.add_argument('--embed_dim', type=int, default=64, metavar='N', help='embedding size')
    parser.add_argument('--lr', type=float, default=0.001, metavar='LR', help='learning rate')
    parser.add_argument('--test_batch_size', type=int, default=1000, metavar='N', help='input batch size for testing')
    parser.add_argument('--epochs', type=int, default=100, metavar='N', help='number of epochs to train')
    args, unknown = parser.parse_known_args()

    os.environ['CUDA_VISIBLE_DEVICES'] = '0'
    use_cuda = False
    if torch.cuda.is_available():
        use_cuda = True
    device = torch.device("cuda" if use_cuda else "cpu")

    embed_dim = args.embed_dim
    dir_data = f'{PROCESSED_FILE_DIR}/FH_Trustworthy_Social_0.6_threshold'

    path_data = dir_data + ".pickle"
    data_file = open(path_data, 'rb')
    history_u_lists, history_ur_lists, history_v_lists, history_vr_lists, train_u, train_v, train_r, test_u, test_v, test_r, social_adj_lists, ratings_list = pickle.load(
        data_file)
    total_num_items = len(set(train_v + test_v))

    trainset = torch.utils.data.TensorDataset(torch.LongTensor(train_u), torch.LongTensor(train_v),
                                              torch.FloatTensor(train_r))
    testset = torch.utils.data.TensorDataset(torch.LongTensor(test_u), torch.LongTensor(test_v),
                                             torch.FloatTensor(test_r))
    train_loader = torch.utils.data.DataLoader(trainset, batch_size=args.batch_size, shuffle=True)
    test_loader = torch.utils.data.DataLoader(testset, batch_size=args.test_batch_size, shuffle=True)
    num_users = history_u_lists.__len__()
    num_items = history_v_lists.__len__()

    u2e = nn.Embedding(num_users, embed_dim).to(device)
    v2e = nn.Embedding(num_items, embed_dim).to(device)
    r2e = nn.Embedding(total_num_items, embed_dim).to(device)  # Define r2e with the correct number of ratings
    total_num_items = len(set(train_v + test_v))

    agg_u_history = UV_Aggregator(v2e, r2e, u2e, embed_dim, cuda=device, uv=True)
    enc_u_history = UV_Encoder(u2e, embed_dim, history_u_lists, history_ur_lists, agg_u_history, cuda=device, uv=True)
    agg_u_social = Social_Aggregator(lambda nodes: enc_u_history(nodes).t(), u2e, embed_dim, cuda=device)
    enc_u = Social_Encoder(lambda nodes: enc_u_history(nodes).t(), embed_dim, social_adj_lists, agg_u_social,
                           base_model=enc_u_history, cuda=device)

    agg_v_history = UV_Aggregator(v2e, r2e, u2e, embed_dim, cuda=device, uv=False)
    enc_v_history = UV_Encoder(v2e, embed_dim, history_v_lists, history_vr_lists, agg_v_history, cuda=device, uv=False)

    graphrec = GraphRec(enc_u, enc_v_history, r2e).to(device)
    optimizer = torch.optim.RMSprop(graphrec.parameters(), lr=args.lr, alpha=0.9)

    for epoch in range(1, args.epochs + 1):
        train(graphrec, device, train_loader, optimizer, epoch, total_num_items)

    top_100_recommendations = get_top_100_recommendations(graphrec, device, test_loader, num_items=total_num_items)

    # Print or use top_100_recommendations as needed
    return top_100_recommendations

if __name__ == "__main__":
    recommendation_list=main()


[1,     0] loss: 0.007
[1,   100] loss: 0.541
[1,   200] loss: 0.514
[1,   300] loss: 0.515
[1,   400] loss: 0.508
[1,   500] loss: 0.503
[1,   600] loss: 0.508
[1,   700] loss: 0.509
[1,   800] loss: 0.503
[2,     0] loss: 0.005
[2,   100] loss: 0.501
[2,   200] loss: 0.500
[2,   300] loss: 0.503
[2,   400] loss: 0.495
[2,   500] loss: 0.500
[2,   600] loss: 0.505
[2,   700] loss: 0.501
[2,   800] loss: 0.498
[3,     0] loss: 0.005
[3,   100] loss: 0.499
[3,   200] loss: 0.502
[3,   300] loss: 0.508
[3,   400] loss: 0.499
[3,   500] loss: 0.498
[3,   600] loss: 0.501
[3,   700] loss: 0.506
[3,   800] loss: 0.505
[4,     0] loss: 0.005
[4,   100] loss: 0.490
[4,   200] loss: 0.499
[4,   300] loss: 0.511
[4,   400] loss: 0.506
[4,   500] loss: 0.495
[4,   600] loss: 0.495
[4,   700] loss: 0.498
[4,   800] loss: 0.503
[5,     0] loss: 0.004
[5,   100] loss: 0.498
[5,   200] loss: 0.502
[5,   300] loss: 0.495
[5,   400] loss: 0.496
[5,   500] loss: 0.503
[5,   600] loss: 0.493
[5,   700] 

In [20]:
file_path = f'{PROCESSED_FILE_DIR}/FH_item_update.pickle'

with open(file_path, 'rb') as file:
    loaded_dictionary = pickle.load(file)

In [21]:
def get_key_by_value(dictionary, value):
    for key, val in dictionary.items():
        if val == value:
            return key
    return None

In [22]:
rows = []
for user, rankings in recommendation_list.items():
    for rank, item in rankings:
        rows.append({"user": user, "rank": rank, "item_update": item, "item": get_key_by_value(loaded_dictionary, item)})

all_recs = pd.DataFrame(rows)

In [23]:
cold_start_item=[2,571]

In [24]:
filtered_recs = all_recs[~all_recs['item'].isin(cold_start_item)]
filtered_recs

,user,rank,item_update,item
0,2314,1,994,1003
1,2314,2,923,931
2,2314,3,791,799
3,2314,4,284,290
4,2314,5,1043,1052
...,...,...,...,...
540695,2530,96,554,562
540696,2530,97,885,893
540697,2530,98,870,878
540698,2530,99,496,503


In [25]:
filtered_recs['rank'] = filtered_recs['rank'].astype(int)

In [26]:
filtered_recs['re-rank'] = filtered_recs.groupby('user')['rank'].rank(method='first')
filtered_recs['rank'] = filtered_recs['re-rank'].astype(int)
filtered_recs=filtered_recs.iloc[:,[0,1,2,3]]

In [27]:
filtered_recs

,user,rank,item_update,item
0,2314,1,994,1003
1,2314,2,923,931
2,2314,3,791,799
3,2314,4,284,290
4,2314,5,1043,1052
...,...,...,...,...
540695,2530,96,554,562
540696,2530,97,885,893
540697,2530,98,870,878
540698,2530,99,496,503


In [28]:
# Define the different top-k values you want to generate
top_k_values = [5, 10, 15, 20]

# Loop through each k value
for k in top_k_values:
    # Filter for top-k recommendations (excluding user 0)
    MA_gnn_all_recs_top_k = filtered_recs[(filtered_recs['rank'] <= k) & (filtered_recs['user'] != 0)]
    
    # Select only the required columns: user, rank, item
    final_all_recs = MA_gnn_all_recs_top_k.iloc[:, [0, 1, 3]]
    
    # Display the final DataFrame (with only 3 columns)
    print(f"\n{'='*60}")
    print(f"TOP-{k} RECOMMENDATIONS (user, rank, item)")
    print(f"{'='*60}")
    display(final_all_recs)
    
    # Save to CSV with k in the filename
    filename = f'{DATA_FILE_DIR}/FH_Trustworthy_Social_0.6_threshold_top_{k}.csv'
    final_all_recs.to_csv(filename, index=False)
    
    print(f"\n✓ Saved {len(final_all_recs)} recommendations to {filename}")
    print(f"  - Number of users: {final_all_recs['user'].nunique()}")


TOP-5 RECOMMENDATIONS (user, rank, item)


,user,rank,item
0,2314,1,1003
1,2314,2,931
2,2314,3,799
3,2314,4,290
4,2314,5,1052
...,...,...,...
540600,2530,1,53
540601,2530,2,805
540602,2530,3,266
540603,2530,4,735



✓ Saved 27030 recommendations to /home/shoaib/recommender-system/royal/journal-revision/New/FH/data/FH_Trustworthy_Social_0.6_threshold_top_5.csv
  - Number of users: 5406

TOP-10 RECOMMENDATIONS (user, rank, item)


,user,rank,item
0,2314,1,1003
1,2314,2,931
2,2314,3,799
3,2314,4,290
4,2314,5,1052
...,...,...,...
540605,2530,6,1192
540606,2530,7,27
540607,2530,8,94
540608,2530,9,277



✓ Saved 54060 recommendations to /home/shoaib/recommender-system/royal/journal-revision/New/FH/data/FH_Trustworthy_Social_0.6_threshold_top_10.csv
  - Number of users: 5406

TOP-15 RECOMMENDATIONS (user, rank, item)


,user,rank,item
0,2314,1,1003
1,2314,2,931
2,2314,3,799
3,2314,4,290
4,2314,5,1052
...,...,...,...
540610,2530,11,739
540611,2530,12,409
540612,2530,13,3
540613,2530,14,738



✓ Saved 81090 recommendations to /home/shoaib/recommender-system/royal/journal-revision/New/FH/data/FH_Trustworthy_Social_0.6_threshold_top_15.csv
  - Number of users: 5406

TOP-20 RECOMMENDATIONS (user, rank, item)


,user,rank,item
0,2314,1,1003
1,2314,2,931
2,2314,3,799
3,2314,4,290
4,2314,5,1052
...,...,...,...
540615,2530,16,1226
540616,2530,17,733
540617,2530,18,29
540618,2530,19,1372



✓ Saved 108120 recommendations to /home/shoaib/recommender-system/royal/journal-revision/New/FH/data/FH_Trustworthy_Social_0.6_threshold_top_20.csv
  - Number of users: 5406


## Evaluation

In [29]:
test=pd.read_csv(f"{DATA_FILE_DIR}/test_ratings_final_before_train.csv")
test=test.iloc[:,[1,2,3]]
test.drop(index=5406,inplace=True)
test

,user,item,ratings
0,0,332,1.0
1,1,66,1.0
2,2,785,1.5
3,3,85,1.5
4,4,120,1.5
...,...,...,...
5401,5401,1,0.5
5402,5402,5,2.0
5403,5403,1,1.0
5404,5404,1,1.5


In [31]:
top_k_values = [5, 10, 15, 20]

# Initialize RecListAnalysis
rla = topn.RecListAnalysis()
rla.add_metric(topn.recip_rank)

# Store results
mrr_results = {}

for k in top_k_values:
    # Load the corresponding CSV
    final_all_recs = pd.read_csv(f"{DATA_FILE_DIR}/FH_Trustworthy_Social_0.6_threshold_top_{k}.csv")
    
    # Compute MRR
    results = rla.compute(final_all_recs, test)
    mean_results = results.mean()
    
    # Check what's in the results
    print(f"\nTop-{k} Results:")
    print(mean_results)
    
    mrr_results[k] = mean_results['recip_rank']

# Create summary DataFrame
mrr_summary = pd.DataFrame(list(mrr_results.items()), columns=['Top-K', 'MRR'])
display(mrr_summary)


Top-5 Results:
nrecs         5.000000
recip_rank    0.009955
dtype: float64

Top-10 Results:
nrecs         10.000000
recip_rank     0.012228
dtype: float64

Top-15 Results:
nrecs         15.000000
recip_rank     0.013952
dtype: float64

Top-20 Results:
nrecs         20.000000
recip_rank     0.015045
dtype: float64


,Top-K,MRR
0,5,0.009955
1,10,0.012228
2,15,0.013952
3,20,0.015045


In [34]:
# Load the key (only once)
key = pd.read_csv(f"{DATA_FILE_DIR}/key_healthstory.csv")
key_news_id = key[["item","label"]].drop_duplicates(subset=["item"], keep="last").reset_index(drop=True)
key_news_id["fake_real_labels"] = key_news_id["label"].apply(lambda x: 0 if x == "fake" else 1)

# Define the MMC function
def mmc(dataframe, top_n):
    """
    Calculates the Misinformation Count (MC) divided by the top-N recommendations for each user.

    Parameters:
    dataframe (DataFrame): DataFrame with columns 'user_id', 'item_id', 'rank', 'news_label'.
    top_n (int): Number of top recommendations to consider.

    Returns:
    float: Mean MC across all users.
    """
    dataframe = dataframe[dataframe['rank'] <= top_n]
    sorted_df = dataframe.sort_values(by=['user', 'rank'])
    misinformation_df = sorted_df[sorted_df['label'] == 'fake']
    user_mc = misinformation_df.groupby('user').size().reset_index(name='M_item')
    user_mc['MC'] = user_mc['M_item'] / top_n
    return user_mc['MC'].mean()

# Compute MC for all top-k values
top_k_values = [5, 10, 15, 20]
mc_results = {}

for k in top_k_values:
    # Load the corresponding CSV
    final_all_recs = pd.read_csv(f"{DATA_FILE_DIR}/FH_Trustworthy_Social_0.6_threshold_top_{k}.csv")
    
    # Merge with news labels
    final_recs_mc = pd.merge(final_all_recs, key_news_id, left_on="item", right_on="item", how='inner')
    
    # Compute MC for this top-k
    mc_score = mmc(final_recs_mc, k)
    mc_results[k] = mc_score
    
    print(f"Top-{k} MC: {mc_score:.4f}")

# Create summary DataFrame
mc_summary = pd.DataFrame(list(mc_results.items()), columns=['Top-K', 'MC'])
display(mc_summary)

Top-5 MC: 0.2587
Top-10 MC: 0.2588
Top-15 MC: 0.2100
Top-20 MC: 0.1890


,Top-K,MC
0,5,0.258661
1,10,0.258783
2,15,0.210014
3,20,0.189012
